In [ ]:
# Fase 4: Evaluación de Recuperación (RAG) - Búsqueda Léxica vs Semántica

**Objetivo:** Comparar el rendimiento del modelo base de recuperación léxica (TF-IDF) frente a la recuperación vectorial semántica (DistilBERT + FAISS). 

Se documentan 5 consultas estratégicas diseñadas para evidenciar cómo la búsqueda semántica comprende el contexto, los sinónimos y la intención del usuario, superando la limitación de la coincidencia exacta de palabras del enfoque léxico.

In [ ]:
# Configuración del entorno y rutas
import sys
from pathlib import Path

# Añadir el directorio raiz al path para poder importar desde src/
root_dir = Path.cwd().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

import torch
import faiss
import pickle
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModel
from IPython.display import display, HTML

from src.data_loader import load_config
from src.lexical_search import LexicalSearcher

# Configuración visual para Pandas
pd.set_option('display.max_colwidth', None)

In [ ]:
# Carga de Modelos y Datos
print("Cargando configuración y recursos...")
params = load_config(root_dir / "config/params.yaml")
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Cargar Transformers (Búsqueda Semántica)
tokenizer = AutoTokenizer.from_pretrained(params["model"]["transformer_name"])
model = AutoModel.from_pretrained(params["model"]["transformer_name"]).to(device).eval()

# 2. Cargar Índice FAISS
index_path = root_dir / params["faiss"]["index_path"]
index = faiss.read_index(str(index_path))

# 3. Cargar Metadata y Motor Léxico
id_map_path = root_dir / params["faiss"]["id_map_path"]
with open(id_map_path, "rb") as f:
    id_map = pickle.load(f)

lexical_searcher = LexicalSearcher(id_map_path)

print("✅ Todos los recursos cargados correctamente.")

In [ ]:
# Función Auxiliar para Codificar y Comparar
def encode_query(text, tokenizer, model, device):
    inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt", max_length=128).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return embeddings / norms

def evaluar_consulta(query, top_k=3):
    print(f"🔍 CONSULTA: '{query}'")
    print("-" * 80)
    
    # 1. Búsqueda Léxica (TF-IDF)
    lex_results, lex_scores = lexical_searcher.search(query, top_k=top_k)
    lex_data = []
    for meta, score in zip(lex_results, lex_scores):
        lex_data.append({
            "Motor": "Léxico (TF-IDF)",
            "Score": round(score, 4),
            "Sentimiento": meta["sentiment_label"],
            "Texto Recuperado": meta["text"][:200] + "..." # Truncamos para mejor visualización
        })
        
    # 2. Búsqueda Semántica (FAISS)
    query_emb = encode_query(query, tokenizer, model, device)
    sem_scores, sem_indices = index.search(query_emb.astype(np.float32), top_k)
    sem_data = []
    for i, idx in enumerate(sem_indices[0]):
        if idx != -1:
            meta = id_map[idx]
            sem_data.append({
                "Motor": "Semántico (FAISS)",
                "Score": round(float(sem_scores[0][i]), 4),
                "Sentimiento": meta["sentiment_label"],
                "Texto Recuperado": meta["text"][:200] + "..."
            })
            
    # Mostrar resultados en tablas de Pandas
    df_lex = pd.DataFrame(lex_data)
    df_sem = pd.DataFrame(sem_data)
    
    display(HTML("<b>Resultados Léxicos (Coincidencia Exacta):</b>"))
    display(df_lex if not df_lex.empty else "No se encontraron coincidencias exactas.")
    
    display(HTML("<br><b>Resultados Semánticos (Contexto y Significado):</b>"))
    display(df_sem)
    print("\n" + "="*80 + "\n")

### Consulta 1: Durabilidad implícita
**Query:** *"stop working after a week"*

**Hipótesis:** El motor léxico buscará literalmente las palabras "stop" o "week". El motor semántico entenderá que el usuario busca productos con mala durabilidad, que se rompieron rápido o resultaron defectuosos.

In [ ]:
evaluar_consulta("stop working after a week")

### Consulta 2: Ergonomía y Salud
**Query:** *"pain in my wrists"*

**Hipótesis:** El léxico dependerá de la aparición exacta de "pain" o "wrists". El modelo vectorial debería conectar el concepto con la ergonomía, recuperando reseñas sobre productos incómodos (uncomfortable, bad posture, stiff) como mouse pads o sillas.

In [ ]:
evaluar_consulta("pain in my wrists")

### Consulta 3: Optimización de Espacios
**Query:** *"perfect for a tiny desk"*

**Hipótesis:** Una búsqueda tradicional fallará si la reseña dice "compact size" o "space saver" en lugar de "tiny". El motor de embeddings mapea estos conceptos al mismo espacio semántico.

In [ ]:
evaluar_consulta("perfect for a tiny desk")

### Consulta 4: Estética y Diseño
**Query:** *"looks very elegant"*

**Hipótesis:** Evaluamos la capacidad del modelo para abstraer atributos visuales. Debería recuperar términos paralelos como "sleek", "premium finish" o "beautiful design", sin forzar la palabra "elegant".

In [ ]:
evaluar_consulta("looks very elegant")

### Consulta 5: Resistencia de Materiales
**Query:** *"cannot hold heavy binders"*

**Hipótesis:** El usuario describe una limitación física de peso. El buscador semántico detectará quejas sobre materiales débiles ("flimsy", "bends under weight", "cheap plastic") en organizadores o estantes, mientras que el léxico se estancará en las palabras aisladas.

In [ ]:
evaluar_consulta("cannot hold heavy binders")

## 📌 Conclusión Técnica de la Evaluación

Tras ejecutar las 5 consultas documentadas, se evidencia empíricamente la ventaja de la arquitectura RAG basada en **Transformers (DistilBERT) + FAISS** sobre los sistemas tradicionales:

1. **Inmunidad al vocabulario estricto:** El modelo léxico (TF-IDF) sufre de *sparse representation*, fallando cuando el usuario y el autor de la reseña usan sinónimos.
2. **Comprensión de conceptos latentes:** El modelo semántico demostró capacidad para agrupar conceptos complejos (Ergonomía, Estética, Durabilidad) sin depender de palabras clave específicas.
3. **Robustez ante el ruido:** En reseñas de Amazon, donde la gramática es irregular, la codificación densa de 768 dimensiones captura la "intención" global del texto de manera mucho más confiable que la frecuencia de términos.